In [8]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage


from dotenv import load_dotenv
load_dotenv()
#os.environ['OPENAI_API_KEY'] = ""

True

In [9]:
import json

input_file = "../dataset/original_formatted/train.json"
output_file = "../dataset/ellipsis_recovered_formatted/train.json"

with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [10]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are given a dialogue. Perform the following transformations:
1. Restore any omitted subjects, objects, or predicates. Example: change "고쳤어" → "[나는 창문을] 고쳤어".
2. Replace pronouns or vague references with specific nouns based on context. Example: change "이것" → "[사과]".
3. If the same speaker speaks consecutively, merge their utterances into one line so that the dialogue alternates between different speakers.
4. Do not transform unnecessary features.
Do not delete any sentence elements that do not fall under the above rules.

Mark only the modified parts with square brackets [ ].
Do not bracket unchanged words.
Keep the rest of the content exactly the same as the original except for the required modifications.



**Few-shot examples (mimic exactly this format)**

Example:
Input:
화자1: 근데 인제 건강이 우리만 막 건강한다고 되는 게 아니잖아. 인제 애들도 건강도 챙겨줘야 되고 그러잖아. 그러니까 너는 애들한테 뭐 따로 먹이는 뭐 식품 뭐 이런 거 있어?
화자2: 유산 유산균하고 비타민 그런데 애들이 잘 안 먹지. 나도 잘 안 먹는 데 애들이 먹나? 근데 쫓아다니면서 챙겨줄 수도 없고 그런데 유산균은 꼭 먹이라 그러더라고. 유산균 장이 건강해야지 전체적으로 다 좋아진다고 그러더라고.
화자2: 그래서 될 수 있으면 비타민도 사놓기는 했는데 그거는 넘어가더라도 나는 유산균은 꼭 먹으라고 얘기한다. 유산균이 어떤 사람은 공복에 먹으라 그러고 어떤 사람은 그냥 아무 때나 먹으라고 하는데 모르겠어. 언제 먹는 게 좋은지
화자2: 그래서 화자1이는 또 천식에 아토피까지 있잖아. 진짜 우리 화자1이는 건강을 누구보다 진짜 걔는 정말 신경 써가지고 지켜야 되는데 지금은 애들 유산균 먹이는 거에 집중하지 유산균이 젤 좋다고는 하더라고. 그 뭐 피부나 아니면 장내 활동이 좋아야지 감기나 뭐 이런 것도 걸렸을 때 빨리 낫는다고.
화자2: 그래서 근데 어떤 유산균이 또 좋은지 몰라. 가루로 된 걸 먹이는 사람도 있고 알약으로 된 걸 먹이는 사람도 있는데 나는 그냥 가루로 된 거 그리고 광고 많이 나오는 거 어~ 나도 거기에 대한 정보가 많이 없어서 그냥 오래된 회사 거? 이런 데 거 먹이고
화자2: 그래서 애들이 건강해야 나도 편하기는 한데 그~ 애들 건강까지 챙겨준다는 것도 어렵고 음~ 지금은 유산균만 먹여. 으 유산균 먹이고 야채 많이 먹이고 그러는데 야채도 잎채소랑 줄기랑 또 다르대. 영양분이 그러니까 뭐를 먹여야지 건강에 더 좋은지 어렵지 다 찾아서 주기가
화자2: 너는 유산균 아직 안 먹였으면 한번 알아봐가지고 특히 화자2이 먼저 먹여. 화자2이가 밥도 잘 안 먹고 하니까 내가 추천해 주는 거 이로울 만한 거는 없는데 한번 알아봐봐. 있을 거야
화자2: 애들 먹이는 거 있어?
화자1: 음~ 나도 유산균하고 종합비타민 뭐 이런 거 먹이는데
화자1: 화자2이는 고3이잖아. 그래서 내가 인제 홍삼 제품을 먹였었어. 근데 어느 날 막 생리통이 너무 심한 거야. 그래가지고 인제 가서 검사를 했지. 그랬더니 혹이 있대. 근데 그~ 홍삼 제품이 여성호르몬이 많잖아.
화자1: 그러니까 그~ 혹에는 여성호르몬 그러니까 홍삼 제품이 안 좋대. 그런다고 나한테 화자2이 무슨 약 먹이는 거 없냐고 물어보더라고. 그래서 이거 쪼금 먹였다 했더니 인제는 먹이지 말라고 하더라고. 그래서 어~ 솔직히 건강해지려고 홍삼을 먹인 거잖아.
화자1: 근데 얘는 또 그걸 먹으면 또 안 된다고 하잖아. 그래서 인제 홍삼은 끊고 유산균하고 뭐 비타민 이런 거 먹고 있는데 솔직히 이런 거 맞춰서 먹이는 것도 너무 힘들어. 뭐에 안 좋은 거 이거 못 먹이고. 근데 누가 그렇게 될 줄 알았나.
화자1: 만 저기 여러 명 중에 한 명 나타난 거겠지만 화자2이는 또 내 생각에 이런 걸 먹여서 또 얘가 이렇게 혹에 났나? 이런 생각도 들고 나 혼자 자책도 하고 뭐 하여튼 그랬는데 건강은 진짜 우리가 한다고 어떻게 할 수 있는 게 아닌 거 같더라고. 그래서
화자1: 지금은 몸에 좋다는 그런 것도 못 먹여. 그~ 고3이라 좀 뭘 좀 먹여주고 싶은데 홍삼 제품 이런 건 전혀 못 먹여. 그래서 사 논 것도 화자2이 아빠가 다 먹었어. 그래서 쪼끔 아까워.
화자1: 너 요새 그~ 폐렴 접종 무슨 그~ 접종 하는 게 되게 많잖아. 여자애들 그 자궁암 뭐 그것도 경부암도 있고 어른들 맞는 폐구균 이런 것도 엄청 많잖아. 그런 거 맞는 거에 대해서 생각해 봤어?
화자2: 진작부터 생각해 봤는데 금액이 만만치 않더라고. 한 번 맞는 거 좋은 거 맞아야 된다는데 기간이 있더라고. 그게 또 1년 가는 게 있고 뭐 2년 가는 게 있고 6개월 짜리가 있고 거기에 따라 금액도 틀려지고 근데 그거 맞는다고 해서 진짜 폐렴이 안 걸릴까? 이런 의구심도 들고 그래서 맞아야 되나 말아야 되나 그러는데
화자2: 우리 시어머니랑 남편은 맞았어. 근데 자기 둘만 맞고 나는 쏙 빼놓고 진짜 그럴 때는 내가 의심이 들어서 안 맞고는 있지만 서운하지. 자기들 맞을 때 나도 좀 맞으라고 하든지.
화자2: 그러니까 어느 병원이 잘하는 곳이 있대. 저렴하면서 name3이가 언제 얘기하더라. 그~ 뭐 혜택 받아서 맞을 수 있는 병원도 있고 이렇다 그러니까 너도 찾아봐가지고 맞을 수 있으면 맞아.
화자2: 나도 인제 안 미루고 맞아는 보려고. 맞아서 소용없으면 소용없는 거고 또 맞아서 좋을 수도 있는 거니까. 그래서 안 걸리면 더 좋은 거고. 가 한번 name3이가 얘기한 대로 혜택 받을 수 있으면 받아가지고 맞아보고.
화자2: 그리고 식구들만 또 그런 거 맞고 다닌다고 그러고 삐져갖고 있을 때가 아니라 나는 내가 알아서 내 건강을 챙겨야 될 거 같아. 나이도 있고 그러니까 근데 진짜 나이 먹으니까 쪼끄만 거 하나에도 서운해지기는 하더라.
화자2: 이게 그런 게 다 건강하고도 이어지는 거 같은데 너는 요새 붓고 이런 거는 없어?
화자1: 으 음~ 한참 붓고 그랬는데 요새는 조금 괜찮긴 하더라고 근데 그것도 나이 들어서 그런 건지 어쩐 건지. 근데 내가 그것도 병원에 가서 검사를 해봐야 되나 이런 생각도 들긴 한데 아직 그것 땜에 병원에 가지는 안 했 안 했고 조금 더 더 부으면 그때 병원에 가볼까 생각 중이야.

Output:
화자 1: 근데 인제 건강이 [가족 전체가 아니라] 우리만 막 건강한다고 되는 게 아니잖아. 인제 [아이들도] 건강도 챙겨줘야 되고 그러잖아. 그러니까 너는 [아이들]한테 뭐 따로 먹이는 [건강기능식품] 뭐 이런 거 있어?
화자 2: [우리 집은] 유산 [즉, 유산균]하고 비타민 [같은 보충제도 있는데] 그런데 [아이들이] 잘 안 먹지. 나도 잘 안 먹는데 [아이들이] 먹나? 근데 쫓아다니면서 챙겨줄 수도 없고 그런데 [그래도] 유산균은 꼭 먹이라 그러더라고. [유산균을 먹어서 장이] 건강해야지 [전체적으로] 다 좋아진다고 그러더라고. 그래서 될 수 있으면 비타민도 사놓기는 했는데 [비타민은] 넘어가더라도 나는 [최소한] 유산균은 꼭 먹으라고 얘기한다. 유산균이 어떤 사람은 공복에 먹으라 그러고 어떤 사람은 그냥 아무 때나 먹으라고 하는데 [정확히] 모르겠어. 언제 먹는 게 좋은지. 그래서 [네 아이] 화자1이는 또 천식에 아토피까지 있잖아. 진짜 [그 아이는] 건강을 누구보다 진짜 [특별히] 신경 써가지고 지켜야 되는데 [나는] 지금은 [아이들] 유산균 먹이는 거에 집중하지 [요즘은] 유산균이 젤 좋다고는 하더라고. 그 뭐 피부나 아니면 [장내 기능이] 좋아야지 감기나 뭐 이런 것도 걸렸을 때 빨리 낫는다고. 그래서 근데 어떤 유산균이 또 좋은지 몰라. 가루로 된 걸 먹이는 사람도 있고 알약으로 된 걸 먹이는 사람도 있는데 나는 그냥 가루로 된 [제품] 그리고 광고 많이 나오는 거 어~ 나도 거기에 대한 정보가 많이 없어서 그냥 오래된 회사 [제품]? 이런 데 [제품] 먹이고. 그래서 [결국] 애들이 건강해야 나도 편하기는 한데 그~ 애들 건강까지 챙겨준다는 것도 어렵고 음~ 지금은 유산균만 먹여. 으 유산균 먹이고 [채소 등] 야채 많이 먹이고 그러는데 야채도 잎채소랑 줄기랑 또 다르대. 영양분이 그러니까 뭐를 먹여야지 건강에 더 좋은지 어렵지 다 찾아서 주기가. 너는 유산균 아직 안 먹였으면 한번 알아봐가지고 특히 [네 둘째] 화자2이 먼저 먹여. 화자2이가 밥도 잘 안 먹고 하니까 내가 추천해 주는 거 [딱히] 이로울 만한 거는 없는데 한번 알아봐봐. 있을 거야. 애들 먹이는 거 있어?
화자 1: 음~ 나도 유산균하고 종합비타민 뭐 이런 거 먹이는데. [그리고] 화자2이는 고3이잖아. 그래서 내가 인제 홍삼 제품을 먹였었어. 근데 어느 날 막 생리통이 너무 심한 거야. 그래가지고 인제 가서 검사를 했지. 그랬더니 [자궁 쪽에] 혹이 있대. 근데 그~ 홍삼 제품이 [식물성] 여성호르몬이 많잖아. 그러니까 그~ [그 아이의] 혹에는 여성호르몬 그러니까 홍삼 제품이 안 좋대. 그런다고 나한테 화자2이 무슨 약 먹이는 거 없냐고 물어보더라고. 그래서 [홍삼 제품을] 이거 쪼금 먹였다 했더니 인제는 먹이지 말라고 하더라고. 그래서 어~ 솔직히 건강해지려고 홍삼을 먹인 거잖아. 근데 [그 아이는] 또 그걸 먹으면 또 안 된다고 하잖아. 그래서 인제 홍삼은 끊고 유산균하고 뭐 비타민 이런 거 먹고 있는데 솔직히 이런 거 맞춰서 먹이는 것도 너무 힘들어. 뭐에 안 좋은 거 이거 못 먹이고. 근데 누가 그렇게 될 줄 알았나. [물론] 저기 여러 명 중에 한 명 나타난 거겠지만 화자2이는 또 내 생각에 이런 걸 먹여서 또 얘가 이렇게 혹에 났나? 이런 생각도 들고 나 혼자 자책도 하고 뭐 하여튼 그랬는데 건강은 진짜 우리가 한다고 어떻게 할 수 있는 게 아닌 거 같더라고. 그래서 지금은 [홍삼 같은] 몸에 좋다는 그런 것도 못 먹여. 그~ 고3이라 좀 뭘 좀 먹여주고 싶은데 홍삼 제품 이런 건 전혀 못 먹여. 그래서 사 논 것도 화자2이 아빠가 다 먹었어. 그래서 쪼끔 아까워. 너 요새 그~ [예방접종 중에서] 폐렴 접종 무슨 그~ 접종 하는 게 되게 많잖아. 여자애들 그 자궁암 뭐 그것도 [자궁경부암(HPV) 백신]도 있고 어른들 맞는 폐구균 이런 것도 엄청 많잖아. 그런 거 맞는 거에 대해서 생각해 봤어?
화자 2: [나도] 진작부터 생각해 봤는데 [접종] 금액이 만만치 않더라고. 한 번 맞는 거 좋은 거 맞아야 된다는데 [백신마다 권장] 기간이 있더라고. 그게 또 1년 가는 게 있고 뭐 2년 가는 게 있고 6개월짜리가 있고 거기에 따라 금액도 틀려지고 근데 그거 맞는다고 해서 진짜 폐렴이 안 걸릴까? 이런 의구심도 들고 그래서 맞아야 되나 말아야 되나 그러는데 우리 시어머니랑 남편은 맞았어. 근데 자기 둘만 맞고 나는 쏙 빼놓고 진짜 그럴 때는 내가 의심이 들어서 안 맞고는 있지만 서운하지. 자기들 맞을 때 나도 좀 맞으라고 하든지. 그러니까 어느 병원이 잘하는 곳이 있대. [상대적으로] 저렴하면서 name3이가 언제 얘기하더라. 그~ 뭐 혜택 받아서 맞을 수 있는 병원도 있고 이렇다 그러니까 너도 찾아봐가지고 맞을 수 있으면 맞아. 나도 인제 안 미루고 맞아는 보려고. 맞아서 소용없으면 소용없는 거고 또 맞아서 좋을 수도 있는 거니까. 그래서 안 걸리면 더 좋은 거고. 가 한번 name3이가 얘기한 대로 혜택 받을 수 있으면 받아가지고 맞아보고. 그리고 [이제는] 식구들만 또 그런 거 맞고 다닌다고 그러고 삐져갖고 있을 때가 아니라 나는 내가 알아서 내 건강을 챙겨야 될 거 같아. 나이도 있고 그러니까 근데 진짜 나이 먹으니까 쪼끄만 거 하나에도 서운해지기는 하더라. 이게 그런 게 다 건강하고도 이어지는 거 같은데 너는 요새 [얼굴이나 다리] 붓고 이런 거는 없어?
화자 1: 으 음~ 한참 붓고 그랬는데 요새는 조금 괜찮긴 하더라고 근데 그것도 나이 들어서 그런 건지 어쩐 건지. 근데 내가 그것도 병원에 가서 검사를 해봐야 되나 이런 생각도 들긴 한데 아직 [그 증상 때문에] 병원에 가지는 안 했 안 했고 조금 더 더 부으면 그때 병원에 가볼까 생각 중이야.

output only the transformed dialogue, nothing else. 
"""

integrated_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt.strip()),
    ("human", "<Actual dialogue to Process>\n{actual_dialogue}")
])

chain = integrated_prompt | llm

In [11]:
# ====== 유틸: LLM이 코드펜스로 감싸서 반환해도 깨끗이 정리 ======
import re

def strip_code_fences(text: str) -> str:
    if text is None:
        return ""
    s = text.strip()
    # ```lang\n...\n``` or ```\n...\n``` 패턴 제거
    m = re.match(r"^```(?:[a-zA-Z0-9_\-]+)?\s*\n(.*)\n```$", s, re.DOTALL)
    if m:
        return m.group(1).strip()
    return s


In [12]:
# ====== 변환 함수: dialogue 1개를 LLM으로 변환(간단 재시도) ======
import time
from typing import Optional

def transform_dialogue_with_llm(dialogue_text: str, max_retries: int = 4, base_wait: float = 2.0) -> str:
    last_err: Optional[Exception] = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = chain.invoke({"actual_dialogue": dialogue_text})
            content = getattr(resp, "content", resp)
            return strip_code_fences(content)
        except Exception as e:
            last_err = e
            wait_s = base_wait * (2 ** (attempt - 1))
            print(f"[경고] 변환 실패 {attempt}/{max_retries}: {e} → {wait_s:.1f}s 후 재시도")
            time.sleep(wait_s)
    print("[경고] 최종 실패: 원본 dialogue를 그대로 사용합니다.")
    return dialogue_text


In [13]:
# ====== 메인 변환 루프: dialogue만 변환, 동일 스키마 유지 ======
from copy import deepcopy
import json
from tqdm import tqdm  # 진행바 라이브러리

converted_items = []
total = len(data)

for item in tqdm(data, desc="Transforming dialogues", ncols=100):
    if not isinstance(item, dict):
        continue
    new_item = deepcopy(item)
    original_dialogue = new_item.get("dialogue", "")
    transformed_dialogue = transform_dialogue_with_llm(original_dialogue)
    new_item["dialogue"] = transformed_dialogue  # 오직 dialogue만 교체
    converted_items.append(new_item)


Transforming dialogues: 100%|███████████████████████████████████| 506/506 [3:18:56<00:00, 23.59s/it]


In [14]:
def dump_one_line_value(value):
    # 배열/객체도 한 줄로 직렬화
    return json.dumps(value, ensure_ascii=False, separators=(',', ':'))

with open(output_file, "w", encoding="utf-8") as f:
    f.write("[\n")
    for i, obj in enumerate(converted_items):
        f.write("  {\n")
        f.write(f'    "id": {dump_one_line_value(obj.get("id"))},\n')
        f.write(f'    "dialogue": {dump_one_line_value(obj.get("dialogue"))},\n')
        f.write(f'    "subject_keyword": {dump_one_line_value(obj.get("subject_keyword"))},\n')
        f.write(f'    "speaker_map": {dump_one_line_value(obj.get("speaker_map"))},\n')
        # 마지막 필드는 콤마 없이
        f.write(f'    "output": {dump_one_line_value(obj.get("output"))}\n')
        f.write("  }" + (",\n" if i < len(converted_items) - 1 else "\n"))
    f.write("]\n")

print(f"[완료] 총 {len(converted_items)}개 항목 저장 (각 데이터 5줄 포맷) → {output_file}")

[완료] 총 506개 항목 저장 (각 데이터 5줄 포맷) → ../dataset/ellipsis_recovered_formatted/train.json
